<a href="https://colab.research.google.com/github/d12eek/ANN-And-DL-Lab-24mcs005/blob/main/ANN_DL_Lab_(BERT).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import torch
from torch import nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizer, BertModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

In [ ]:
file_path ='/content/IMDB Dataset.csv'
df = pd.read_csv(file_path, on_bad_lines='skip', quoting=3)

In [ ]:
df.head()

,,,,,,,,,,,,,,,,,,,,,,,,,,review,sentiment
"""One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right",as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence,which set in right from the word GO. Trust me,this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs,sex or violence. Its is hardcore,in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City,an experimental section of the prison where all the cells have glass fronts and face inwards,so privacy is not high on the agenda. Em City is home to many..Aryans,Muslims,gangstas,Latinos,Christians,Italians,Irish and more....so scuffles,death stares,dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fact that it goes where other shows wouldn't dare. Forget pretty pictures painted for mainstream audiences,forget charm,forget romance...OZ doesn't mess around. The first episode I ever saw struck me as so nasty it was surreal,I couldn't say I was ready for it,but as I watched more,I developed a taste for Oz,and got accustomed to the high levels of graphic violence. Not just violence,but injustice (crooked guards who'll be sold out for a nickel,inmates who'll kill on order and get away with it,well mannered,middle class inmates being turned into prison bitches due to their lack of street skills or prison experience) Watching Oz,you may become comfortable with what is uncom...,positive
"""A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting",and sometimes discomforting,"sense of realism to the entire piece. <br /><br />The actors are extremely well chosen- Michael Sheen not only """"has got all the polari"""" but he has all the voices down pat too! You can truly see the seamless editing guided by the references to Williams' diary entries",not only is it well worth the watching but it is a terrificly written and performed piece. A masterful production about one of the great master's of comedy and his life. <br /><br />The realism really comes home with the little things: the fantasy of the guard which,rather than use the traditional 'dream' techniques remains solid then disappears. It plays on our knowledge and our senses,"particularly with the scenes concerning Orton and Halliwell and the sets (particularly of their flat with Halliwell's murals decorating every surface) are terribly well done.""",positive,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
"""I thought this was a wonderful way to spend time on a too hot summer weekend",sitting in the air conditioned theater and watching a light-hearted comedy. The plot is simplistic,but the dialogue is witty and the characters are likable (even the well bread suspected serial killer). While some may be disappointed when they realize this is not Match Point 2: Risk Addiction,I thought it was proof that Woody Allen is still fully in control of the style many of us have grown to love.<br /><br />This was the most I'd laughed at one of Woody's comedies in years (dare I say a decade?). While I've never been impressed with Scarlet Johanson,"in this she managed to tone down her """"sexy"""" image and jumped right into a average",but spirited young woman.<br /><br />This may not be the crown jewel of his career,"but it was wittier than """"Devil Wears Prada"""" and more interesting than """"Superman"""" a great comedy to go see with friends.""",positive,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
"""Basically there's a family where a little boy (Jake) thinks there's a zombie in his closet & his parents are fighting all the time.<b

In [ ]:
df.shape

(7977, 2)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 7977 entries, ('"One of the other reviewers has mentioned that after watching just 1 Oz episode you\'ll be hooked. They are right', ' as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence', ' which set in right from the word GO. Trust me', ' this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs', ' sex or violence. Its is hardcore', ' in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City', ' an experimental section of the prison where all the cells have glass fronts and face inwards', ' so privacy is not high on the agenda. Em City is home to many..Aryans', ' Muslims', ' gangstas', ' Latinos', ' Christians', ' Italians', ' Irish and more....so scuffles', ' death stares', " dodgy deali

In [ ]:
df['sentiment'].value_counts()

,count
sentiment,
negative,34
positive,30


In [ ]:
df['review_sentiment']=df['sentiment'].apply(lambda x: 1 if x=='positive' else 0)
df.sample(5)

,,,,,,,,,,,,,,,,,,,,,,,,,,review,sentiment,review_sentiment
"""DD films were damn corny",damn stupid and had a plot which seemed wafer thin but those days they was a plot at least<br /><br />This film isn't just a comedy but a mix of melodrama,romance everything<br /><br />Every drama scene is blown out of proportion<br /><br />The comedy is funny but corny too Yet the film keeps you entertained,those days Govinda films were loud,crass yet they had some funny moments people enjoyed<br /><br />David Dhawan does a okay job Music is okay<br /><br />Govinda acts well in comedy and drama Karisma is decent in parts and annoys in parts Kader is as usual Gulshan,"Prem Chopra are typecast Shakti is hilarious""",negative,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
"""I just started watching The Show around July. I found it by mistake",I was channel surfing during a Vacation. It is a great show,"I just wish it wasn't on so late at night. It's on at 12:30 AM. As a working person it makes it hard to watch all the time.<br /><br />I read some comments. I did not agree with the late one about not growing up in the 60's and not believing that this stuff can happen.<br /><br />I grew up in the 60's. I'm Hispanic and I had a """"White"""" boyfriend plus we had black friends in High School. I believe people get along because of their interests and personalities and it has nothing to do with being a certain race or color.<br /><br />I can't wait till the show goes on DVD so I can buy it. This way I can see it from the beginning.""",positive,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
"""Eye in the Labyrinth is not your average Giallo...and to be honest",I'm not really sure that it really is a Giallo; but Giallo or not,despite some problems,this is certainly a very interesting little film. I'm hesitant to call it a Giallo because the film doesn't feature most of the things that make these films what they are; but many genre entries break the mould,and this would seem to be one of them. The film doesn't feature any brutal murders as many Giallo's do,but this is made up for with a surreal atmosphere and a plot just about confusing enough to remain interesting for the duration. The plot seems simple enough in that it focuses on a doctor who is murdered by Julie,his patient who,for some reason,she sees him as her lover and father and is offended when he walks out on her. We then relocate to a big house lived in by a number of people,but nothing is really what it seems as there are a number of secrets surrounding various events that happened before Julie's arrival...<br /><br />The film seems to be professing something about how the mind is like a labyrinth. This never really comes off,and I preferred to just sit back and enjoy what was going on rather than worrying about what point (if any) the film is trying to make. Eye in the Labyrinth is directed by Mario Caiano,the director behind the excellent Night of the Doomed some years earlier. He doesn't create the atmosphere as well in this film as he did in the earlier one; but the surreal aspects of the story come off well,and the mystery is always kept up which stops the film from becoming boring. The film stars Rosemary Dexter,who provides eye candy throughout and also delivers a good performance. Most of the rest of the cast aren't really worth mentioning,with the exceptions of Adolfo Celi,who is good as the villain of the piece and Alida Valli,whom cult fans will remember from a whole host of excellent cult flicks. The film does explain itself at the end; which is lucky as I'm sure I'm not the only viewer who was more than a little confused by then! Overall,"this may not be classic stuff; but its good enough and worth seeing.""",positive,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
"""I was looking for a documentary of the same journalistic quality as Frontline or """"Fog of War"""" (by Errol Morris). Instead I 

In [ ]:
df.isnull().sum()

,0
review,7850
sentiment,7913
review_sentiment,0


In [ ]:
df.isna().sum()

,0
review,7850
sentiment,7913
review_sentiment,0


In [ ]:
texts = df['review'].tolist()
labels = [1 if sentiment == "positive" else 0 for sentiment in df['sentiment'].tolist()]

Create a custom dataset class for text classification

In [ ]:
class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        # Convert text to string if it is not already
        if not isinstance(text, str):
            text = str(text)
        label = self.labels[idx]
        encoding = self.tokenizer(text, return_tensors='pt', max_length=self.max_length, padding='max_length', truncation=True)
        return {'input_ids': encoding['input_ids'].flatten(), 'attention_mask': encoding['attention_mask'].flatten(), 'label': torch.tensor(label)}

In [ ]:
# BERT Classifier Model
class BERTClassifier(nn.Module):
    def __init__(self, bert_model_name, num_classes):
        super(BERTClassifier, self).__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        self.dropout = nn.Dropout(0.1)
        self.fc = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output
        x = self.dropout(pooled_output)
        logits = self.fc(x)
        return logits

In [ ]:
# Training Function
def train(model, data_loader, optimizer, scheduler, device):
    model.train()
    for batch in data_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = nn.CrossEntropyLoss()(outputs, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()


In [ ]:
# Evaluation Function
def evaluate(model, data_loader, device):
    model.eval()
    predictions = []
    actual_labels = []
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            _, preds = torch.max(outputs, dim=1)
            predictions.extend(preds.cpu().tolist())
            actual_labels.extend(labels.cpu().tolist())
    return accuracy_score(actual_labels, predictions), classification_report(actual_labels, predictions)

In [ ]:
def predict_sentiment(text, model, tokenizer, device, max_length=128):
    model.eval()
    encoding = tokenizer(text, return_tensors='pt', max_length=max_length, padding='max_length', truncation=True)
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        _, preds = torch.max(outputs, dim=1)
        return "positive" if preds.item() == 1 else "negative"

In [ ]:
# Main Execution
if __name__ == "__main__":
    # Parameters
    bert_model_name = 'bert-base-uncased'
    num_classes = 2
    max_length = 128
    batch_size = 16
    num_epochs = 3
    learning_rate = 2e-5

In [ ]:
train_texts, val_texts, train_labels, val_labels = train_test_split(texts, labels, test_size=0.2, random_state=42)


In [ ]:
tokenizer = BertTokenizer.from_pretrained(bert_model_name)
train_dataset = TextClassificationDataset(train_texts, train_labels, tokenizer, max_length)
val_dataset = TextClassificationDataset(val_texts, val_labels, tokenizer, max_length)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BERTClassifier(bert_model_name, num_classes).to(device)
optimizer = AdamW(model.parameters(), lr=learning_rate)
total_steps = len(train_dataloader) * num_epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

In [ ]:
# Training Loop
for epoch in range(num_epochs):
        print(f"Epoch {epoch + 1}/{num_epochs}")
        train(model, train_dataloader, optimizer, scheduler, device)
        accuracy, report = evaluate(model, val_dataloader, device)
        print(f"Validation Accuracy: {accuracy:.4f}")
        print(report)

Epoch 1/3


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Validation Accuracy: 0.9962
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1590
           1       0.00      0.00      0.00         6

    accuracy                           1.00      1596
   macro avg       0.50      0.50      0.50      1596
weighted avg       0.99      1.00      0.99      1596

Epoch 2/3
Validation Accuracy: 0.9975
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1590
           1       0.60      1.00      0.75         6

    accuracy                           1.00      1596
   macro avg       0.80      1.00      0.87      1596
weighted avg       1.00      1.00      1.00      1596

Epoch 3/3
Validation Accuracy: 0.9969
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1590
           1       0.67      0.33      0.44         6

    accuracy                           1.00      1596
   macro avg       0.83 

In [ ]:
torch.save(model.state_dict(), "bert_classifier.pth")

In [ ]:
test_text = "The movie was great and I really enjoyed the performances of the actors."
sentiment = predict_sentiment(test_text, model, tokenizer, device)
print("The movie was great and I really enjoyed the performances of the actors.")
print(f"Predicted sentiment: {sentiment}")

The movie was great and I really enjoyed the performances of the actors.
Predicted sentiment: positive
